In [3]:
from pathlib import Path
import contextlib
import os
import sys

import h5py
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from omegaconf import OmegaConf


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "model_training" / "trainer.py").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing model_training/trainer.py")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
MODEL_TRAINING_ROOT = REPO_ROOT / "model_training"

if str(MODEL_TRAINING_ROOT) not in sys.path:
    sys.path.insert(0, str(MODEL_TRAINING_ROOT))

from helpers.dataset import BrainToTextDataset, train_test_split_indicies
from helpers.data_augmentations import gauss_smooth
from helpers.evaluate_model_helpers import LOGIT_TO_PHONEME
from models.rnn_model import GRUDecoder
from snn_backbone import SpikingModel


# Point this at the checkpoint directory you want to evaluate.
CHECKPOINT_DIR = (
    REPO_ROOT / "model_training/logs/rnn/2026-04-03_17-52-31_037802_rnn_pid114858/checkpoints"
).resolve()
OUTPUT_PATH = CHECKPOINT_DIR / "validation_only_ctc_export.pt"
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0


def resolve_from_model_training(path_like) -> Path:
    path = Path(str(path_like)).expanduser()
    return path if path.is_absolute() else (MODEL_TRAINING_ROOT / path).resolve()


def load_run_args(checkpoint_dir: Path):
    args_path = checkpoint_dir / "args.yaml"
    if not args_path.exists():
        raise FileNotFoundError(f"Missing args.yaml at {args_path}")

    args = OmegaConf.load(args_path)
    args.dataset.dataset_dir = str(resolve_from_model_training(args.dataset.dataset_dir))
    return args


def build_trials_and_val_loader(args):
    dataset_dir = Path(args["dataset"]["dataset_dir"])
    sessions = list(args["dataset"]["sessions"])
    seed = int(args["dataset"]["seed"])

    bad_trials_dict = args["dataset"].get("bad_trials_dict")
    if bad_trials_dict is not None:
        bad_trials_dict = OmegaConf.to_container(bad_trials_dict, resolve=True)

    train_files = [str(dataset_dir / session / "data_train.hdf5") for session in sessions]
    val_files = [str(dataset_dir / session / "data_val.hdf5") for session in sessions]

    train_trials, _ = train_test_split_indicies(
        file_paths=train_files,
        test_percentage=0,
        seed=seed,
        bad_trials_dict=bad_trials_dict,
    )
    _, val_trials = train_test_split_indicies(
        file_paths=val_files,
        test_percentage=1,
        seed=seed,
        bad_trials_dict=bad_trials_dict,
    )

    feature_subset = args["dataset"].get("feature_subset")
    feature_subset = None if feature_subset is None else list(feature_subset)

    val_dataset = BrainToTextDataset(
        trial_indicies=val_trials,
        split="test",
        days_per_batch=None,
        n_batches=None,
        batch_size=int(args["dataset"]["batch_size"]),
        must_include_days=None,
        random_seed=seed,
        feature_subset=feature_subset,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=None,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )
    return train_trials, val_loader


def build_model(args):
    common = dict(
        neural_dim=int(args["model"]["n_input_features"]),
        n_days=len(args["dataset"]["sessions"]),
        n_classes=int(args["dataset"]["n_classes"]),
        input_dropout=float(args["model"]["input_network"]["input_layer_dropout"]),
        patch_size=int(args["model"]["patch_size"]),
        patch_stride=int(args["model"]["patch_stride"]),
    )

    if args["model_type"] == "rnn":
        return GRUDecoder(
            n_units=int(args["model"]["n_units"]),
            rnn_dropout=float(args["model"]["rnn_dropout"]),
            n_layers=int(args["model"]["n_layers"]),
            **common,
        )

    if args["model_type"] == "snn":
        session_adaptation_cfg = args.get("session_adaptation")
        if session_adaptation_cfg is None:
            session_adaptation_cfg = args.get("session_threshold_adapter", {})
        return SpikingModel(
            cfg=args["snn_config"],
            debug=False,
            session_adaptation_cfg=session_adaptation_cfg,
            **common,
        )

    raise ValueError(f"Unsupported model_type: {args['model_type']}")


def load_checkpoint_weights(model, checkpoint_dir: Path):
    checkpoint_path = checkpoint_dir / "best_checkpoint"
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing checkpoint at {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    state_dict = checkpoint["model_state_dict"]

    cleaned_state_dict = {}
    for key, value in state_dict.items():
        clean_key = key.replace("module.", "").replace("_orig_mod.", "")
        cleaned_state_dict[clean_key] = value

    model.load_state_dict(cleaned_state_dict, strict=True)
    return checkpoint_path


def compute_output_lengths(args, n_time_steps: torch.Tensor) -> torch.Tensor:
    patch_size = int(args["model"]["patch_size"])
    patch_stride = int(args["model"]["patch_stride"])

    if patch_size > 0 and patch_stride > 0:
        output_lengths = (n_time_steps - patch_size) // patch_stride + 1
    else:
        output_lengths = n_time_steps

    return torch.clamp(output_lengths, min=1).to(torch.int32)


def apply_validation_transform(args, features: torch.Tensor, device: torch.device):
    transform_args = args["dataset"]["data_transforms"]
    if transform_args["smooth_data"]:
        features = gauss_smooth(
            inputs=features,
            device=device,
            smooth_kernel_std=transform_args["smooth_kernel_std"],
            smooth_kernel_size=transform_args["smooth_kernel_size"],
        )
    return features


def build_session_summary_map(args, train_trials):
    session_adaptation_cfg = args.get("session_adaptation")
    if session_adaptation_cfg is None:
        session_adaptation_cfg = args.get("session_threshold_adapter", {})

    adaptation_mode = session_adaptation_cfg.get("adaptation_mode", "day")
    if adaptation_mode != "session_threshold":
        return None

    feature_subset = args["dataset"].get("feature_subset")
    feature_subset = None if feature_subset is None else list(feature_subset)

    summary_by_id = {}
    for session_id, session_info in train_trials.items():
        total_sum = None
        total_sq_sum = None
        total_count = 0

        with h5py.File(session_info["session_path"], "r") as handle:
            for trial_idx in session_info["trials"]:
                trial_group = handle[f"trial_{trial_idx:04d}"]
                input_features = trial_group["input_features"][:]

                if feature_subset is not None:
                    input_features = input_features[:, feature_subset]

                input_features = input_features.astype("float64", copy=False)
                if total_sum is None:
                    n_channels = int(input_features.shape[1])
                    total_sum = torch.zeros(n_channels, dtype=torch.float64)
                    total_sq_sum = torch.zeros(n_channels, dtype=torch.float64)

                trial_tensor = torch.from_numpy(input_features)
                total_sum += trial_tensor.sum(dim=0)
                total_sq_sum += (trial_tensor ** 2).sum(dim=0)
                total_count += int(input_features.shape[0])

        if total_sum is None or total_count == 0:
            raise ValueError(f"No usable input_features found for session_id={session_id}")

        mean = total_sum / total_count
        var = torch.clamp((total_sq_sum / total_count) - (mean ** 2), min=0.0)
        std = torch.sqrt(var)
        summary_by_id[int(session_id)] = torch.cat([mean, std], dim=0).to(torch.float32)

    return summary_by_id


def lookup_session_summaries(session_summary_map, session_ids: torch.Tensor, device: torch.device):
    if not session_summary_map:
        return None
    summaries = [session_summary_map[int(session_id)] for session_id in session_ids.cpu().tolist()]
    return torch.stack(summaries, dim=0).to(device)


def maybe_autocast(device: torch.device, use_amp: bool):
    if device.type == "cuda" and use_amp:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return contextlib.nullcontext()


def forward_model(args, model, features, day_indices, session_summaries):
    if args["model_type"] == "snn":
        return model(features, day_indices, session_summary=session_summaries)
    return model(features, day_indices)


def ctc_greedy_decode(framewise_argmax: torch.Tensor) -> torch.Tensor:
    decoded = torch.unique_consecutive(framewise_argmax, dim=0)
    decoded = decoded[decoded != 0]
    return decoded.to(torch.int64)


def pad_and_stack_1d(sequences, pad_value: int):
    max_len = max(seq.numel() for seq in sequences)
    out = torch.full((len(sequences), max_len), pad_value, dtype=torch.int64)
    for i, seq in enumerate(sequences):
        if seq.numel() > 0:
            out[i, : seq.numel()] = seq
    return out


def pad_and_stack_2d(sequences, pad_value: float):
    max_time = max(seq.shape[0] for seq in sequences)
    feature_dim = sequences[0].shape[1]
    out = torch.full((len(sequences), max_time, feature_dim), pad_value, dtype=sequences[0].dtype)
    for i, seq in enumerate(sequences):
        out[i, : seq.shape[0]] = seq
    return out


def run_validation_export(checkpoint_dir: Path, output_path: Path):
    args = load_run_args(checkpoint_dir)
    train_trials, val_loader = build_trials_and_val_loader(args)
    session_summary_map = build_session_summary_map(args, train_trials)

    model = build_model(args)
    checkpoint_path = load_checkpoint_weights(model, checkpoint_dir)
    model.to(DEVICE)
    model.eval()

    dataset_probability_val = args["dataset"].get("dataset_probability_val")
    if dataset_probability_val is None:
        dataset_probability_val = [1] * len(args["dataset"]["sessions"])

    all_logits = []
    all_probs = []
    all_argmax = []
    all_decoded = []
    all_targets = []
    all_input_lengths = []
    all_target_lengths = []

    with torch.inference_mode():
        for batch in tqdm(val_loader, desc="Validation export", unit="batch"):
            day = int(batch["day_indicies"][0].item())
            if int(dataset_probability_val[day]) == 0:
                continue

            features = batch["input_features"].to(DEVICE, non_blocking=True).float()
            labels = batch["seq_class_ids"].to(DEVICE, non_blocking=True).long()
            n_time_steps = batch["n_time_steps"].to(DEVICE, non_blocking=True)
            target_lengths = batch["phone_seq_lens"].to(DEVICE, non_blocking=True)
            day_indices = batch["day_indicies"].to(DEVICE, non_blocking=True)
            session_ids = batch["session_ids"].to(DEVICE, non_blocking=True)

            with maybe_autocast(DEVICE, bool(args["use_amp"])):
                features = apply_validation_transform(args, features, DEVICE)
                input_lengths = compute_output_lengths(args, n_time_steps)
                session_summaries = lookup_session_summaries(session_summary_map, session_ids, DEVICE)

                logits = forward_model(args, model, features, day_indices, session_summaries)
                input_lengths = torch.clamp(input_lengths, max=logits.shape[1])

                probs = torch.softmax(logits, dim=-1)
                argmax_predictions = torch.argmax(logits, dim=-1)

            for i in range(logits.shape[0]):
                valid_len = int(input_lengths[i].item())
                target_len = int(target_lengths[i].item())

                sample_logits = logits[i, :valid_len].detach().cpu().to(torch.float32)
                sample_probs = probs[i, :valid_len].detach().cpu().to(torch.float32)
                sample_argmax = argmax_predictions[i, :valid_len].detach().cpu().to(torch.int64)
                sample_decoded = ctc_greedy_decode(sample_argmax)
                sample_target = labels[i, :target_len].detach().cpu().to(torch.int64)

                all_logits.append(sample_logits)
                all_probs.append(sample_probs)
                all_argmax.append(sample_argmax)
                all_decoded.append(sample_decoded)
                all_targets.append(sample_target)
                all_input_lengths.append(valid_len)
                all_target_lengths.append(target_len)

    export = {
        "logits": pad_and_stack_2d(all_logits, pad_value=0.0),
        "probabilities": pad_and_stack_2d(all_probs, pad_value=0.0),
        "argmax_predictions": pad_and_stack_1d(all_argmax, pad_value=-1),
        "decoded_ctc_sequences": pad_and_stack_1d(all_decoded, pad_value=-1),
        "target_sequences": pad_and_stack_1d(all_targets, pad_value=-1),
        "input_lengths": torch.tensor(all_input_lengths, dtype=torch.int32),
        "target_lengths": torch.tensor(all_target_lengths, dtype=torch.int32),
        "phoneme_vocab": LOGIT_TO_PHONEME,
        "pad_values": {
            "argmax_predictions": -1,
            "decoded_ctc_sequences": -1,
            "target_sequences": -1,
            "logits": 0.0,
            "probabilities": 0.0,
        },
        "checkpoint_path": str(checkpoint_path),
        "args_path": str(checkpoint_dir / "args.yaml"),
        "device": str(DEVICE),
    }

    output_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(export, output_path)

    for key in [
        "logits",
        "probabilities",
        "argmax_predictions",
        "decoded_ctc_sequences",
        "target_sequences",
        "input_lengths",
        "target_lengths",
    ]:
        print(f"{key}: {tuple(export[key].shape)}")
    print(f"output_file: {output_path}")

    return export


_ = run_validation_export(CHECKPOINT_DIR, OUTPUT_PATH)


/tmp/ipykernel_187842/1640607647.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location="cpu")


Validation export:   0%|          | 0/108 [00:00<?, ?batch/s]

logits: (1426, 593, 41)
probabilities: (1426, 593, 41)
argmax_predictions: (1426, 593)
decoded_ctc_sequences: (1426, 67)
target_sequences: (1426, 67)
input_lengths: (1426,)
target_lengths: (1426,)
output_file: /home/alex/snn_for_bts/brain_to_text/model_training/logs/rnn/2026-04-03_17-52-31_037802_rnn_pid114858/checkpoints/validation_only_ctc_export.pt
